In [5]:
import numpy as np
import tvm
from tvm.relay import expr as _expr
indices = np.array([3,4])
indices = _expr.const(indices)
print(indices)

meta[relay.Constant][0]



In [1]:
import torch  
import torch.nn as nn

# 创建一个示例张量  
x = torch.range(0,26).view(1, 3, 3, 3) 
print("Original tensor:")  
print(x)  
  
# 使用 unfold 展开张量  
unfold = nn.Unfold(kernel_size=(2, 2))  
unfolded = unfold(x)
print("\nUnfolded tensor:")  
print(unfolded)  
print("Unfolded tensor shape:", unfolded.shape)

/tmp/ipykernel_13461/1363731462.py:4: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  x = torch.range(0,26).view(1, 3, 3, 3)


Original tensor:
tensor([[[[ 0.,  1.,  2.],
          [ 3.,  4.,  5.],
          [ 6.,  7.,  8.]],

         [[ 9., 10., 11.],
          [12., 13., 14.],
          [15., 16., 17.]],

         [[18., 19., 20.],
          [21., 22., 23.],
          [24., 25., 26.]]]])

Unfolded tensor:
tensor([[[ 0.,  1.,  3.,  4.],
         [ 1.,  2.,  4.,  5.],
         [ 3.,  4.,  6.,  7.],
         [ 4.,  5.,  7.,  8.],
         [ 9., 10., 12., 13.],
         [10., 11., 13., 14.],
         [12., 13., 15., 16.],
         [13., 14., 16., 17.],
         [18., 19., 21., 22.],
         [19., 20., 22., 23.],
         [21., 22., 24., 25.],
         [22., 23., 25., 26.]]])
Unfolded tensor shape: torch.Size([1, 12, 4])


In [1]:
n = 1
c = 3
h = 3
w = 3
up_num = n*c*h*w
x = torch.range(0, up_num - 1).view(n, c, h, w) 
# print("Original tensor:")  
# print(x)  
k = (2,2)
p = (0,0)
d = (1,1)
s = (1,1)
on = 1
ow = k[0] * k[1]
oc1 = (h + 2 * p[0] - (k[0] + 2 *(d[0] - 1))) // s[0] + 1
oc2 = (w + 2 * p[1] - (k[1] + 2 *(d[1] - 1))) // s[1] + 1
oc = c * oc1 * oc2
output = torch.zeros([on,oc,ow])
for i in range(0,on):
    for j in range(0,oc):
        for m in range(0,ow):
            # print(i,j,m)
            n_index = i
            c_index = j // (oc1 * oc2)
            h_index = j % (oc1 * oc2) // oc2 * s[0] + (m // k[0])*d[0]
            w_index = j % (oc1 * oc2) % oc2 * s[1] + (m % k[1])*d[1]
            output[i][j][m] = x[n_index][c_index][h_index][w_index]
            
# 使用 unfold 展开张量  
unfold = nn.Unfold(kernel_size = k,
        dilation = d,
        padding = p,
        stride = s)  
unfolded = unfold(x)
print((output == unfolded).all())

NameError: name 'torch' is not defined

In [2]:
import tvm
from tvm import relay
def get_demo_mod():
    d1 = relay.var("d1", shape=(1, 32, 56, 56), dtype="float32")
    w1 = relay.var("w1", shape=(32, 32, 3, 3), dtype="float32")
    b1 = relay.var("b1", shape=(32,), dtype="float32")
    conv = relay.nn.conv2d(d1, w1, strides=(1, 1), padding=(1, 1))
    bias = relay.nn.bias_add(conv, b1)
    relu = relay.nn.relu(bias)

    func = relay.Function([d1, w1, b1], relu)
    mod = tvm.IRModule.from_expr(func)
    mod = relay.transform.InferType()(mod)
    return mod

# Without DNNL
mod = get_demo_mod()
print(mod)

with tvm.transform.PassContext(opt_level=2):
    graph, lib, params = relay.build(mod, target="c", params=None)
    print(lib.get_source())

def @main(%d1: Tensor[(1, 32, 56, 56), float32] /* ty=Tensor[(1, 32, 56, 56), float32] */, %w1: Tensor[(32, 32, 3, 3), float32] /* ty=Tensor[(32, 32, 3, 3), float32] */, %b1: Tensor[(32), float32] /* ty=Tensor[(32), float32] */) -> Tensor[(1, 32, 56, 56), float32] {
  %0 = nn.conv2d(%d1, %w1, padding=[1, 1, 1, 1]) /* ty=Tensor[(1, 32, 56, 56), float32] */;
  %1 = nn.bias_add(%0, %b1) /* ty=Tensor[(1, 32, 56, 56), float32] */;
  nn.relu(%1) /* ty=Tensor[(1, 32, 56, 56), float32] */
}



One or more operators have not been tuned. Please tune your model for better performance. Use DEBUG logging level to see more details.


// tvm target: c -keys=cpu 
#define TVM_EXPORTS
#include "tvm/runtime/c_runtime_api.h"
#include "tvm/runtime/c_backend_api.h"
#include <math.h>
#include <stdbool.h>
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t tvmgen_default_fused_nn_conv2d_nn_bias_add_nn_relu(void* args, int32_t* arg_type_ids, int32_t num_args, void* out_ret_value, int32_t* out_ret_tcode, void* resource_handle);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t tvmgen_default_fused_nn_conv2d_nn_bias_add_nn_relu(void* args, int32_t* arg_type_ids, int32_t num_args, void* out_ret_value, int32_t* out_ret_tcode, void* resource_handle) {
  int32_t p0_code = arg_type_ids[0];
  int32_t p1_code = arg_type_ids[1];
  int32_t p2_code = arg_type_ids[2];
  int32_t T_relu_code = arg_type_ids[3];
  void* p0 = (((TVMValue*)args)[0].v_handle);
  void* p1 = (((TVMValue*)args)[1].v_handle);
  void* p2 = (((TVMValue*)args)[2].v_handle);
  void* T_relu = (((TVMValue*)args)[3].v_handle);
  void* tvmgen_default_fused_nn_conv2d_nn_

C:\Users\senli\AppData\Local\Temp\ipykernel_8864\1878239930.py:21: DeprecationWarning: legacy graph executor behavior of producing json / lib / params will be removed in the next release. Please see documents of tvm.contrib.graph_executor.GraphModule for the  new recommended usage.
  graph, lib, params = relay.build(mod, target="c", params=None)


In [4]:
import torch
import torch.nn as nn
from tvm import relay
from tvm.contrib import graph_executor
class AddModel(nn.Module):
    def __init__(self,):
        super(AddModel, self).__init__()
        self.a = torch.ones([1,2,3])
    def forward(self, v_0):
        one_hot = v_0 + self.a
        return one_hot

model = AddModel()
model.eval()
input = torch.ones([1,2,3])
mod = torch.jit.trace(model, input)
shape_list = [('x', (1, 2, 3))]
mod, params = relay.frontend.from_pytorch(mod, shape_list)
print(mod)

RuntimeError: Numpy is not available

In [8]:
# import tvm
# print(tvm.target.Target.list_kinds())
import torch
# def split_by_last_comma(s):  
#     # 找到最后一个逗号的索引  
#     last_comma_index = s.rfind(',')  
      
#     # 如果找到了逗号  
#     if last_comma_index != -1:  
#         # 使用逗号索引+1来分割字符串，这样可以确保逗号不被包含在任一部分中  
#         # 索引+1是因为我们要的是逗号后面的部分  
#         return s[:last_comma_index], s[last_comma_index+1:]  
#     else:  
#         # 如果没有找到逗号，则返回整个字符串和空字符串  
#         return s, ""
    
# s = 'name_hint: x, shape: (1, 3, 224, 224), dtype: float32'
# a = s.split(':')
# aaa = []
# for aa in a:
#     s1, s2 = split_by_last_comma(aa)
#     if s1 != '':
#         aaa.append(s1.strip())
#     if s2 != '':
#         aaa.append(s2.strip())
# print(aaa)
# assert len(aaa) %2 == 0
print(type(torch.float32))

<class 'torch.dtype'>
